In [ ]:
# 运行此代码单元格来挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

# 成功后 Google Drive内容将可以在以下路径访问：
# '/content/drive/My Drive/'
# 假如文件夹名为 'my_project_data' 并且存储在Google Drive的根目录下
# 在Colab中就可以通过 '/content/drive/My Drive/my_project_data/' 来访问它。
drive_path = '/content/drive/My Drive/自选题' # 定义想要切换到的文件夹路径

if os.path.exists(drive_path):
    os.chdir(drive_path)
    print(f"当前工作目录已切换至: {os.getcwd()}")
else:
    print(f"错误: 路径不存在 {drive_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
当前工作目录已切换至: /content/drive/My Drive/自选题


# Experiment: EMCAD Full Training On ETIS

这一册是 EMCAD baseline 的唯一完整定义来源，并且已经改为真正使用你放在 `data/ETIS/` 下的 polyp segmentation 数据。


## 加载基础工具

这里只加载最基础的环境和结果保存工具，真实数据流、模型和训练逻辑都在本 notebook 内定义。


In [ ]:
from pathlib import Path
import json
import os
import random
import sys

from scripts.project_utils import (
    ARTIFACTS,
    DATA_ROOT,
    ETIS_ROOT,
    PVT_PRETRAINED_ROOT,
    ensure_project_dirs,
    get_default_project_config,
    load_project_config,
    print_env_summary,
    save_json,
    set_seed,
    try_import_torch,
)

ensure_project_dirs()
set_seed()
torch = try_import_torch()
ENV_SUMMARY = print_env_summary(torch)
ROOT = Path.cwd().resolve()


{
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "root": "/content/drive/My Drive/自选题",
  "data_root": "/content/drive/My Drive/自选题/data",
  "etis_root": "/content/drive/My Drive/自选题/data/ETIS",
  "pvt_pretrained_root": "/content/drive/My Drive/自选题/data/pvt_pretrained_pth",
  "torch_installed": true,
  "cuda_available": true
}


## 读取共享实验配置

这一段直接读取 `00` 已写入的统一项目配置，因此后面的 notebook 不再重复维护同一份 `PROJECT_CONFIG`。


In [ ]:
PROJECT_CONFIG = load_project_config()
PROJECT_CONFIG


{'dataset': 'ETIS',
 'task': 'Polyp Segmentation',
 'paper_repo': 'https://github.com/SLDGroup/EMCAD',
 'baseline_repos': {'Swin-Unet': 'https://github.com/HuCaoFighting/Swin-Unet',
  'U-Net': 'https://github.com/milesial/Pytorch-UNet'},
 'emcad_scale': 'PVT-EMCAD-B0',
 'metrics': ['Dice'],
 'fixed_visual_sample': '100.png',
 'etis_paths': {'root': '/content/drive/My Drive/自选题/data/ETIS',
  'train_images': '/content/drive/My Drive/自选题/data/ETIS/train/images',
  'train_masks': '/content/drive/My Drive/自选题/data/ETIS/train/masks',
  'val_images': '/content/drive/My Drive/自选题/data/ETIS/val/images',
  'val_masks': '/content/drive/My Drive/自选题/data/ETIS/val/masks',
  'test_images': '/content/drive/My Drive/自选题/data/ETIS/test/images',
  'test_masks': '/content/drive/My Drive/自选题/data/ETIS/test/masks',
  'train_list': '/content/drive/My Drive/自选题/data/ETIS/train_list_etis.txt',
  'val_list': '/content/drive/My Drive/自选题/data/ETIS/val_list_etis.txt',
  'test_list': '/content/drive/My Drive/自选题/

## 定义 ETIS 数据管线与训练评估工具

下面的代码块提供 ETIS 的数据读取、Dice 评估、训练循环、测试评估和可视化导出函数。


In [ ]:
assert torch is not None, "需要先安装 PyTorch 才能运行本 notebook。"

import math
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

TRAIN_IMAGE_DIR = ETIS_ROOT / "train" / "images"
TRAIN_MASK_DIR = ETIS_ROOT / "train" / "masks"
VAL_IMAGE_DIR = ETIS_ROOT / "val" / "images"
VAL_MASK_DIR = ETIS_ROOT / "val" / "masks"
TEST_IMAGE_DIR = ETIS_ROOT / "test" / "images"
TEST_MASK_DIR = ETIS_ROOT / "test" / "masks"
TRAIN_LIST_PATH = ETIS_ROOT / "train_list_etis.txt"
VAL_LIST_PATH = ETIS_ROOT / "val_list_etis.txt"
TEST_LIST_PATH = ETIS_ROOT / "test_list_etis.txt"
PVT_B0_PATH = PVT_PRETRAINED_ROOT / "pvt_v2_b0.pth"

def read_list_file(path):
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def default_visual_sample():
    items = read_list_file(TEST_LIST_PATH)
    return items[0] if items else None

def load_rgb_mask(image_path, mask_path, image_size):
    image = Image.open(image_path).convert("RGB").resize((image_size, image_size), Image.BILINEAR)
    mask = Image.open(mask_path).convert("L").resize((image_size, image_size), Image.NEAREST)
    image = np.array(image, dtype=np.float32) / 255.0
    image = (image - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array([0.229, 0.224, 0.225], dtype=np.float32)
    mask = (np.array(mask, dtype=np.float32) > 127).astype(np.float32)
    image = torch.from_numpy(image.transpose(2, 0, 1))
    mask = torch.from_numpy(mask).unsqueeze(0)
    return image, mask

class ETISSegmentationDataset(Dataset):
    def __init__(self, split, image_size=352):
        self.split = split
        self.image_size = image_size
        self.image_dir = ETIS_ROOT / split / "images"
        self.mask_dir = ETIS_ROOT / split / "masks"
        self.items = read_list_file(ETIS_ROOT / f"{split}_list_etis.txt")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        filename = self.items[idx]
        image_path = self.image_dir / filename
        mask_path = self.mask_dir / filename
        image, mask = load_rgb_mask(image_path, mask_path, self.image_size)
        return {
            "name": filename,
            "image": image,
            "mask": mask,
        }

def build_dataloaders(cfg):
    train_dataset = ETISSegmentationDataset("train", image_size=cfg["image_size"])
    val_dataset = ETISSegmentationDataset("val", image_size=cfg["image_size"])
    test_dataset = ETISSegmentationDataset("test", image_size=cfg["image_size"])
    train_loader = DataLoader(train_dataset, batch_size=cfg["batch_size"], shuffle=True, num_workers=cfg["num_workers"])
    val_loader = DataLoader(val_dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=cfg["num_workers"])
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=cfg["num_workers"])
    return train_loader, val_loader, test_loader

def dice_from_logits(logits, mask, eps=1e-6):
    prediction = (torch.sigmoid(logits) > 0.5).float()
    intersection = (prediction * mask).sum(dim=(1, 2, 3))
    union = prediction.sum(dim=(1, 2, 3)) + mask.sum(dim=(1, 2, 3))
    score = (2 * intersection + eps) / (union + eps)
    return score.mean()

class DiceLoss(nn.Module):
    def forward(self, logits, mask):
        probability = torch.sigmoid(logits)
        intersection = (probability * mask).sum(dim=(1, 2, 3))
        union = probability.sum(dim=(1, 2, 3)) + mask.sum(dim=(1, 2, 3))
        loss = 1 - (2 * intersection + 1e-6) / (union + 1e-6)
        return loss.mean()

class SegmentationCriterion(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits, mask):
        return self.bce(logits, mask) + self.dice(logits, mask)

def run_epoch(model, loader, criterion, optimizer=None, device="cpu"):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    total_dice = 0.0
    steps = 0
    for batch in loader:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)
        logits = model(images)
        loss = criterion(logits, masks)
        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item())
        total_dice += float(dice_from_logits(logits, masks).item())
        steps += 1
    return {
        "loss": total_loss / max(steps, 1),
        "dice": total_dice / max(steps, 1),
    }

def train_model(model, train_loader, val_loader, cfg, device="cpu"):
    criterion = SegmentationCriterion()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
    best_state = None
    best_dice = -1.0
    history = []
    for epoch in range(cfg["epochs"]):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer=optimizer, device=device)
        val_metrics = run_epoch(model, val_loader, criterion, optimizer=None, device=device)
        record = {
            "epoch": epoch + 1,
            "train_loss": round(train_metrics["loss"], 4),
            "train_dice": round(train_metrics["dice"], 4),
            "val_loss": round(val_metrics["loss"], 4),
            "val_dice": round(val_metrics["dice"], 4),
        }
        history.append(record)
        print(record)
        if val_metrics["dice"] > best_dice:
            best_dice = val_metrics["dice"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return history, round(best_dice, 4)

def evaluate_loader(model, loader, device="cpu"):
    criterion = SegmentationCriterion()
    summary = run_epoch(model, loader, criterion, optimizer=None, device=device)
    summary = {
        "loss": round(summary["loss"], 4),
        "dice": round(summary["dice"], 4),
    }
    per_sample = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            image = batch["image"].to(device)
            mask = batch["mask"].to(device)
            logits = model(image)
            per_sample.append(
                {
                    "name": batch["name"][0],
                    "dice": round(float(dice_from_logits(logits, mask).item()), 4),
                }
            )
    return summary, per_sample

def export_visualization(model, sample_name, image_size, save_path, device="cpu"):
    image_path = TEST_IMAGE_DIR / sample_name
    mask_path = TEST_MASK_DIR / sample_name
    image, mask = load_rgb_mask(image_path, mask_path, image_size)
    model.eval()
    with torch.no_grad():
        logits = model(image.unsqueeze(0).to(device))
        prediction = (torch.sigmoid(logits).squeeze(0).squeeze(0).cpu().numpy() > 0.5).astype(np.float32)
    image_np = image.permute(1, 2, 0).cpu().numpy()
    image_np = image_np * np.array([0.229, 0.224, 0.225], dtype=np.float32) + np.array([0.485, 0.456, 0.406], dtype=np.float32)
    image_np = np.clip(image_np, 0.0, 1.0)
    mask_np = mask.squeeze(0).cpu().numpy()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image_np)
    axes[0].set_title(f"{sample_name} image")
    axes[1].imshow(mask_np, cmap="gray")
    axes[1].set_title("ground truth")
    axes[2].imshow(prediction, cmap="gray")
    axes[2].set_title("prediction")
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    return str(save_path)

def save_history_figure(history, save_path):
    epochs = [item["epoch"] for item in history]
    train_dice = [item["train_dice"] for item in history]
    val_dice = [item["val_dice"] for item in history]
    train_loss = [item["train_loss"] for item in history]
    val_loss = [item["val_loss"] for item in history]
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(epochs, train_dice, label="train dice")
    axes[0].plot(epochs, val_dice, label="val dice")
    axes[0].set_title("Dice")
    axes[0].legend()
    axes[1].plot(epochs, train_loss, label="train loss")
    axes[1].plot(epochs, val_loss, label="val loss")
    axes[1].set_title("Loss")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    return str(save_path)


## 定义 EMCAD baseline 结构

下面的代码块给出本项目中 EMCAD baseline 的唯一完整实现来源，并显式支持加载 `pvt_v2_b0.pth`。


In [ ]:
assert torch is not None, "需要先安装 PyTorch 才能运行本 notebook。"

class DWConv(nn.Module):
    def __init__(self, dim=768):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, groups=dim)

    def forward(self, x, h, w):
        b, n, c = x.shape
        x = x.transpose(1, 2).reshape(b, c, h, w)
        x = self.dwconv(x)
        return x.flatten(2).transpose(1, 2)

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.0):
        super().__init__()
        hidden_features = hidden_features or in_features
        out_features = out_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.dwconv = DWConv(hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x, h, w):
        x = self.fc1(x)
        x = self.dwconv(x, h, w)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        return self.drop(x)

class Attention(nn.Module):
    def __init__(self, dim, num_heads=1, sr_ratio=1):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.q = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, dim * 2)
        self.proj = nn.Linear(dim, dim)
        self.sr_ratio = sr_ratio
        if sr_ratio > 1:
            self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio)
            self.norm = nn.LayerNorm(dim)
        else:
            self.sr = None
            self.norm = None

    def forward(self, x, h, w):
        b, n, c = x.shape
        q = self.q(x).reshape(b, n, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        source = x
        if self.sr is not None:
            source = source.transpose(1, 2).reshape(b, c, h, w)
            source = self.sr(source).reshape(b, c, -1).transpose(1, 2)
            source = self.norm(source)
        kv = self.kv(source).reshape(b, -1, 2, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(b, n, c)
        return self.proj(out)

class PVTBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, sr_ratio=1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads=num_heads, sr_ratio=sr_ratio)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = Mlp(dim, hidden_features=int(dim * mlp_ratio))

    def forward(self, x, h, w):
        x = x + self.attn(self.norm1(x), h, w)
        x = x + self.mlp(self.norm2(x), h, w)
        return x

class PatchEmbed(nn.Module):
    def __init__(self, in_chans, embed_dim, kernel_size, stride, padding):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=kernel_size, stride=stride, padding=padding)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        h, w = x.shape[-2:]
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x, h, w

class PVTv2B0Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed1 = PatchEmbed(3, 32, kernel_size=7, stride=4, padding=3)
        self.patch_embed2 = PatchEmbed(32, 64, kernel_size=3, stride=2, padding=1)
        self.patch_embed3 = PatchEmbed(64, 160, kernel_size=3, stride=2, padding=1)
        self.patch_embed4 = PatchEmbed(160, 256, kernel_size=3, stride=2, padding=1)
        self.block1 = nn.ModuleList([PVTBlock(32, num_heads=1, mlp_ratio=8, sr_ratio=8) for _ in range(2)])
        self.block2 = nn.ModuleList([PVTBlock(64, num_heads=2, mlp_ratio=8, sr_ratio=4) for _ in range(2)])
        self.block3 = nn.ModuleList([PVTBlock(160, num_heads=5, sr_ratio=2) for _ in range(2)])
        self.block4 = nn.ModuleList([PVTBlock(256, num_heads=8, sr_ratio=1) for _ in range(2)])
        self.norm1 = nn.LayerNorm(32)
        self.norm2 = nn.LayerNorm(64)
        self.norm3 = nn.LayerNorm(160)
        self.norm4 = nn.LayerNorm(256)

    def forward(self, x):
        x1, h1, w1 = self.patch_embed1(x)
        for block in self.block1:
            x1 = block(x1, h1, w1)
        x1 = self.norm1(x1)
        x1_map = x1.transpose(1, 2).reshape(x1.shape[0], 32, h1, w1)

        x2, h2, w2 = self.patch_embed2(x1_map)
        for block in self.block2:
            x2 = block(x2, h2, w2)
        x2 = self.norm2(x2)
        x2_map = x2.transpose(1, 2).reshape(x2.shape[0], 64, h2, w2)

        x3, h3, w3 = self.patch_embed3(x2_map)
        for block in self.block3:
            x3 = block(x3, h3, w3)
        x3 = self.norm3(x3)
        x3_map = x3.transpose(1, 2).reshape(x3.shape[0], 160, h3, w3)

        x4, h4, w4 = self.patch_embed4(x3_map)
        for block in self.block4:
            x4 = block(x4, h4, w4)
        x4 = self.norm4(x4)
        x4_map = x4.transpose(1, 2).reshape(x4.shape[0], 256, h4, w4)
        return x1_map, x2_map, x3_map, x4_map

    def load_pretrained(self, weight_path):
        assert weight_path.exists(), f"未找到预训练权重: {weight_path}"
        state = torch.load(weight_path, map_location="cpu")
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        missing, unexpected = self.load_state_dict(state, strict=False)
        print({"loaded_weight_path": str(weight_path), "missing_keys": len(missing), "unexpected_keys": len(unexpected)})

class ConvNormAct(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)

class MSCAM(nn.Module):
    def __init__(self, channels, kernel_sizes=(1, 3, 5)):
        super().__init__()
        self.branches = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Conv2d(channels, channels, kernel_size=k, padding=k // 2, bias=False),
                    nn.BatchNorm2d(channels),
                    nn.GELU(),
                )
                for k in kernel_sizes
            ]
        )
        self.proj = nn.Conv2d(channels * len(kernel_sizes), channels, kernel_size=1, bias=False)

    def forward(self, x):
        return self.proj(torch.cat([branch(x) for branch in self.branches], dim=1))

class CAB(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, hidden, kernel_size=1)
        self.fc2 = nn.Conv2d(hidden, channels, kernel_size=1)

    def forward(self, x):
        weights = self.pool(x)
        weights = F.gelu(self.fc1(weights))
        weights = torch.sigmoid(self.fc2(weights))
        return x * weights

class SAB(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.GELU(),
            nn.Conv2d(channels, 1, kernel_size=1),
        )

    def forward(self, x):
        return x * torch.sigmoid(self.conv(x))

class EMCADDecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, kernel_sizes=(1, 3, 5)):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.fuse = ConvNormAct(in_channels + skip_channels, out_channels, kernel_size=1)
        self.ms_cam = MSCAM(out_channels, kernel_sizes=kernel_sizes)
        self.cab = CAB(out_channels)
        self.sab = SAB(out_channels)
        self.refine = ConvNormAct(out_channels, out_channels, kernel_size=3)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.fuse(x)
        x = self.ms_cam(x)
        x = self.cab(x)
        x = self.sab(x)
        return self.refine(x)

class EMCADBaseline(nn.Module):
    def __init__(self, kernel_sizes=(1, 3, 5)):
        super().__init__()
        self.encoder = PVTv2B0Encoder()
        self.dec3 = EMCADDecoderBlock(256, 160, 160, kernel_sizes=kernel_sizes)
        self.dec2 = EMCADDecoderBlock(160, 64, 64, kernel_sizes=kernel_sizes)
        self.dec1 = EMCADDecoderBlock(64, 32, 32, kernel_sizes=kernel_sizes)
        self.head = nn.Conv2d(32, 1, kernel_size=1)

    def load_pretrained_backbone(self, weight_path):
        self.encoder.load_pretrained(weight_path)

    def forward(self, x):
        skip1, skip2, skip3, bottleneck = self.encoder(x)
        x = self.dec3(bottleneck, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)
        logits = self.head(x)
        if logits.shape[-2:] != x.shape[-2:]:
            logits = F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return F.interpolate(logits, scale_factor=4, mode="bilinear", align_corners=False)


## 构建 ETIS dataloader 并检查预训练权重

这里会读取 `train / val / test` 三段数据，并检查 `data/pvt_pretrained_pth/pvt_v2_b0.pth` 是否存在。


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = PROJECT_CONFIG["train"]
train_loader, val_loader, test_loader = build_dataloaders(cfg)
weight_path = Path(PROJECT_CONFIG["pvt_pretrained_path"])
assert weight_path.exists(), f"未找到预训练权重: {weight_path}"
{
    "device": device,
    "train_count": len(train_loader.dataset),
    "val_count": len(val_loader.dataset),
    "test_count": len(test_loader.dataset),
    "fixed_visual_sample": default_visual_sample(),
    "pretrained_path": str(weight_path),
}


{'device': 'cuda',
 'train_count': 156,
 'val_count': 20,
 'test_count': 20,
 'fixed_visual_sample': '100.png',
 'pretrained_path': '/content/drive/My Drive/自选题/data/pvt_pretrained_pth/pvt_v2_b0.pth'}

## 实例化 EMCAD baseline 并加载 B0 预训练权重

这里显式加载你已准备好的 PVTv2-B0 权重，然后把 EMCAD 送到当前可用设备。


In [ ]:
model = EMCADBaseline().to(device)
model.load_pretrained_backbone(weight_path)


{'loaded_weight_path': '/content/drive/My Drive/自选题/data/pvt_pretrained_pth/pvt_v2_b0.pth', 'missing_keys': 0, 'unexpected_keys': 2}


## 运行完整训练与验证流程

这里按统一配置完成完整训练，并保留验证集上表现最好的模型权重。


In [16]:
history, best_val_dice = train_model(model, train_loader, val_loader, cfg, device=device)
best_val_dice


{'epoch': 1, 'train_loss': 1.5079, 'train_dice': 0.1855, 'val_loss': 1.6674, 'val_dice': 0.0852}
{'epoch': 2, 'train_loss': 1.3924, 'train_dice': 0.374, 'val_loss': 1.5964, 'val_dice': 0.227}
{'epoch': 3, 'train_loss': 1.3226, 'train_dice': 0.496, 'val_loss': 1.3632, 'val_dice': 0.4675}
{'epoch': 4, 'train_loss': 1.2845, 'train_dice': 0.5761, 'val_loss': 1.2813, 'val_dice': 0.5246}
{'epoch': 5, 'train_loss': 1.2587, 'train_dice': 0.6497, 'val_loss': 1.2417, 'val_dice': 0.6406}
{'epoch': 6, 'train_loss': 1.2341, 'train_dice': 0.6831, 'val_loss': 1.2377, 'val_dice': 0.5855}
{'epoch': 7, 'train_loss': 1.2254, 'train_dice': 0.7059, 'val_loss': 1.2334, 'val_dice': 0.6545}
{'epoch': 8, 'train_loss': 1.2095, 'train_dice': 0.7175, 'val_loss': 1.208, 'val_dice': 0.6608}
{'epoch': 9, 'train_loss': 1.1926, 'train_dice': 0.7554, 'val_loss': 1.2027, 'val_dice': 0.6411}
{'epoch': 10, 'train_loss': 1.1751, 'train_dice': 0.7682, 'val_loss': 1.1909, 'val_dice': 0.6736}
{'epoch': 11, 'train_loss': 1.160

0.8128

## 在验证集和测试集上评估 EMCAD

这里统一输出验证集摘要、测试集摘要和逐样本 Dice，方便后续对照、消融和改进读取。


In [17]:
val_summary, val_rows = evaluate_loader(model, val_loader, device=device)
test_summary, test_rows = evaluate_loader(model, test_loader, device=device)
val_summary, test_summary


({'loss': 0.8054, 'dice': 0.8128}, {'loss': 0.6782, 'dice': 0.8787})

## 导出训练曲线和统一测试样本可视化

这里把 EMCAD 的训练曲线和固定测试样本预测图导出到 artifacts，后续所有比较都引用这些产物。


In [18]:
history_figure_path = ARTIFACTS / "figures" / "emcad_training_history.png"
saved_history_figure = save_history_figure(history, history_figure_path)
visual_sample = default_visual_sample()
visual_path = ARTIFACTS / "figures" / "emcad_visual_sample.png"
saved_visual = export_visualization(model, visual_sample, cfg["image_size"], visual_path, device=device)
saved_history_figure, saved_visual


('/content/drive/My Drive/自选题/artifacts/figures/emcad_training_history.png',
 '/content/drive/My Drive/自选题/artifacts/figures/emcad_visual_sample.png')

## 保存 EMCAD baseline 结果

这里输出的是后续所有对照、消融和改进实验要引用的 baseline 结果来源。


In [19]:
checkpoint_path = ARTIFACTS / "checkpoints" / "emcad_b0_etis_best.pth"
torch.save(model.state_dict(), checkpoint_path)

save_json(
    {
        "dataset": "ETIS",
        "emcad_scale": PROJECT_CONFIG["emcad_scale"],
        "device": device,
        "pretrained_path": str(weight_path),
        "history": history,
        "best_val_dice": best_val_dice,
        "val_summary": val_summary,
        "test_summary": test_summary,
        "val_rows": val_rows,
        "test_rows": test_rows,
        "visual_sample": visual_sample,
        "visual_path": saved_visual,
        "history_figure_path": saved_history_figure,
        "checkpoint_path": str(checkpoint_path),
    },
    ARTIFACTS / "records" / "emcad_etis_results.json",
)
